In [2]:
pip install holidays

  Using cached holidays-0.93-py3-none-any.whl.metadata (51 kB)
Using cached holidays-0.93-py3-none-any.whl (1.4 MB)
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [18]:
import pandas as pd
import boto3
import io
import numpy as np
import gc
import holidays
from datetime import datetime, timedelta

print("Libraries loaded.")

Libraries loaded.


In [19]:
# reading the sheets from xlsx
# connect to S3
s3 = boto3.client('s3')
bucket = 'synchrony-callcenter'

# read the workbook
obj = s3.get_object(Bucket=bucket, Key='raw/Datathon_Data.xlsx')
file = io.BytesIO(obj['Body'].read())

# load each sheet by exact name
daily_A = pd.read_excel(file, sheet_name='A - Daily')
interval_A = pd.read_excel(file, sheet_name='A - Interval')
daily_B = pd.read_excel(file, sheet_name='B - Daily')
interval_B = pd.read_excel(file, sheet_name='B - Interval')
daily_C= pd.read_excel(file, sheet_name='C - Daily')
interval_C = pd.read_excel(file, sheet_name='C - Interval')
daily_D= pd.read_excel(file, sheet_name='D - Daily')
interval_D = pd.read_excel(file, sheet_name='D - Interval')

In [20]:
sheets = ['A - Daily', 'A - Interval', 'B - Daily', 'B - Interval',
          'C - Daily', 'C - Interval', 'D - Daily', 'D - Interval',
          'Daily Staffing']

dfs = {}
for sheet in sheets:
    file.seek(0)
    dfs[sheet] = pd.read_excel(file, sheet_name=sheet)

print("All sheets loaded.")
for sheet, df in dfs.items():
    print(f"  {sheet}: {df.shape}")

All sheets loaded.
  A - Daily: (731, 5)
  A - Interval: (4083, 8)
  B - Daily: (731, 5)
  B - Interval: (4293, 8)
  C - Daily: (731, 5)
  C - Interval: (4368, 8)
  D - Daily: (731, 5)
  D - Interval: (4358, 8)
  Daily Staffing: (365, 5)


In [21]:
# print 5 rows from each sheet with shape
for sheet_name, df in dfs.items():
    print(f"Sheet: {sheet_name} | Shape: {df.shape}")
    print(df.head())

Sheet: A - Daily | Shape: (731, 5)
           Date  Call Volume     CCT  Service Level  Abandon Rate
0  01/01/24 Mon       2147.0  302.45         0.9855        0.0037
1  01/02/24 Tue       7458.0  349.22         0.5213        0.1136
2  01/03/24 Wed       6882.0  331.07         0.7046        0.0432
3  01/04/24 Thu       6208.0  341.80         0.7200        0.0403
4  01/05/24 Fri       6190.0  334.56         0.8063        0.0291
Sheet: A - Interval | Shape: (4083, 8)
   Month  Day  Interval  Service Level  Call Volume  Abandoned Calls  \
0  April    1  00:00:00            1.0          5.0              0.0   
1  April    1  00:30:00            1.0          5.0              0.0   
2  April    1  01:00:00            1.0          4.0              0.0   
3  April    1  01:30:00            1.0          3.0              0.0   
4  April    1  02:00:00            1.0          1.0              0.0   

   Abandoned Rate     CCT  
0             0.0  137.60  
1             0.0  263.40  
2            

In [22]:
#  Preprocess Daily Staffing first (needed by all centers)
staffing = dfs['Daily Staffing'].copy()

# rename unnamed first column to date
staffing.rename(columns={staffing.columns[0]: 'date'}, inplace=True)

# parse date column
staffing['date'] = pd.to_datetime(staffing['date'])

# rename center columns explicitly
staffing.rename(columns={
    'A': 'staffing_A',
    'B': 'staffing_B',
    'C': 'staffing_C',
    'D': 'staffing_D'
}, inplace=True)

# sort and reset index
staffing.sort_values('date', inplace=True)
staffing.reset_index(drop=True, inplace=True)

print("Staffing sheet cleaned.")
print(f"Shape: {staffing.shape}")
print(f"Date range: {staffing['date'].min()} to {staffing['date'].max()}")
print(f"Missing values:\n{staffing.isnull().sum()}")
print(staffing.head())

Staffing sheet cleaned.
Shape: (365, 5)
Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00
Missing values:
date           0
staffing_A    27
staffing_B    35
staffing_C    37
staffing_D    36
dtype: int64
        date  staffing_A  staffing_B  staffing_C  staffing_D
0 2025-01-01        47.0        75.0       353.0       143.0
1 2025-01-02        82.0       184.0       491.0       195.0
2 2025-01-03        92.0       186.0       462.0       183.0
3 2025-01-04        70.0       148.0       352.0       155.0
4 2025-01-05        40.0       110.0       224.0        98.0


In [23]:
# check raw column names and first 3 rows for all daily sheets
for center in ['A', 'B', 'C', 'D']:
    raw = dfs[f'{center} - Daily']
    print(f"Center {center} - Daily | Columns: {list(raw.columns)}")
    print(raw.head(3).to_string())

Center A - Daily | Columns: ['Date', 'Call Volume', 'CCT', 'Service Level', 'Abandon Rate']
           Date  Call Volume     CCT  Service Level  Abandon Rate
0  01/01/24 Mon       2147.0  302.45         0.9855        0.0037
1  01/02/24 Tue       7458.0  349.22         0.5213        0.1136
2  01/03/24 Wed       6882.0  331.07         0.7046        0.0432
Center B - Daily | Columns: ['Date', 'Call Volume', 'CCT', 'Service Level', 'Abandon Rate']
           Date  Call Volume     CCT  Service Level  Abandon Rate
0  01/01/24 Mon       4406.0  301.02         0.9785        0.0057
1  01/02/24 Tue      14914.0  356.37         0.5650        0.1320
2  01/03/24 Wed      14132.0  347.41         0.6218        0.0797
Center C - Daily | Columns: ['Date', 'Call Volume', 'CCT', 'Service Level', 'Abandon Rate']
           Date  Call Volume     CCT  Service Level  Abandon Rate
0  01/01/24 Mon       8059.0  333.48         0.9984        0.0014
1  01/02/24 Tue      30738.0  370.36         0.5464        0.100

In [24]:
# check raw column names and first 3 rows for all interval sheets
for center in ['A', 'B', 'C', 'D']:
    raw = dfs[f'{center} - Interval']
    print(f"Center {center} - Interval | Columns: {list(raw.columns)}")
    print(raw.head(3).to_string())

Center A - Interval | Columns: ['Month', 'Day', 'Interval', 'Service Level', 'Call Volume', 'Abandoned Calls', 'Abandoned Rate', 'CCT']
   Month  Day  Interval  Service Level  Call Volume  Abandoned Calls  Abandoned Rate     CCT
0  April    1  00:00:00            1.0          5.0              0.0             0.0  137.60
1  April    1  00:30:00            1.0          5.0              0.0             0.0  263.40
2  April    1  01:00:00            1.0          4.0              0.0             0.0  333.25
Center B - Interval | Columns: ['Month', 'Day', 'Interval', 'Service Level', 'Call Volume', 'Abandoned Calls', 'Abandoned Rate', 'CCT']
   Month  Day  Interval  Service Level  Call Volume  Abandoned Calls  Abandoned Rate     CCT
0  April    1  00:00:00         1.0000         22.0              0.0             0.0  480.05
1  April    1  00:30:00         0.9524         21.0              0.0             0.0  301.71
2  April    1  01:00:00         1.0000          8.0              0.0         

In [32]:
# Define the Daily preprocessing function
us_holidays = holidays.US(years=[2024, 2025])

def preprocess_daily(df, center_id):
    df = df.copy()
    audit_dropped = []
    audit_imputed = []

    # Step 1: Rename columns 
    df.columns = ['date_raw', 'call_volume', 'CCT', 'service_level', 'abandon_rate']

    # Step 2: Parse date - format is mm/dd/yy Day e.g. "01/01/24 Mon" 
    df['date'] = pd.to_datetime(
        df['date_raw'].astype(str).str.split(' ').str[0],
        format='%m/%d/%y',
        errors='coerce'
    )
    df['day_of_week'] = df['date'].dt.dayofweek

    # Step 3: Call Volume - cast to numeric (no commas in raw data) 
    df['call_volume'] = pd.to_numeric(df['call_volume'], errors='coerce')

    # Step 4: CCT - cast to float 
    df['CCT'] = pd.to_numeric(df['CCT'], errors='coerce')

    # Step 5: Service Level and Abandon Rate - already 0.0-1.0 decimals 
    # just ensure numeric, no divide needed
    df['service_level'] = pd.to_numeric(df['service_level'], errors='coerce')
    df['abandon_rate'] = pd.to_numeric(df['abandon_rate'],  errors='coerce')

    # Step 6: Drop rows where date could not be parsed 
    bad_date = df['date'].isna()
    if bad_date.sum() > 0:
        audit_dropped.append(pd.DataFrame({
            'date': df.loc[bad_date, 'date_raw'],
            'reason': 'unparseable date',
            'center': center_id,
            'sheet': 'daily'
        }))
        df = df[~bad_date].copy()

    # Step 7: Sort by date
    df.sort_values('date', inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Step 8: Drop duplicate dates 
    dupes = df.duplicated(subset='date', keep='first')
    if dupes.sum() > 0:
        print(f"  [{center_id}] Dropping {dupes.sum()} duplicate date rows")
        df = df[~dupes].copy()
        df.reset_index(drop=True, inplace=True)

    metric_cols = ['call_volume', 'CCT', 'service_level', 'abandon_rate']

    # Step 9: Case 1 - all metrics null -drop
    all_null = df[metric_cols].isnull().all(axis=1)
    if all_null.sum() > 0:
        audit_dropped.append(pd.DataFrame({
            'date' : df.loc[all_null, 'date'],
            'reason': 'all metrics null',
            'center': center_id,
            'sheet': 'daily'
        }))
        df = df[~all_null].copy()
        df.reset_index(drop=True, inplace=True)

    # Step 10: Add imputed_flag 
    df['imputed_flag'] = 0

    # Step 11: Case 2 - 1-2 metrics null - linear interpolation 
    null_counts = df[metric_cols].isnull().sum(axis=1)
    case2_mask= (null_counts >= 1) & (null_counts <= 2)
    if case2_mask.sum() > 0:
        df = df.set_index('date')
        for col in metric_cols:
            null_before = df[col].isnull().sum()
            df[col]= df[col].interpolate(method='time')
            null_after = df[col].isnull().sum()
            if null_before > null_after:
                audit_imputed.append({
                    'col' : col,
                    'method': 'linear interpolation',
                    'count':null_before - null_after,
                    'center': center_id
                })
        df.reset_index(inplace=True)
        df.loc[case2_mask, 'imputed_flag'] = 1

    # Step 12: Case 3 - 3-4 metrics null → same-weekday avg (3 before + 3 after)
    null_counts = df[metric_cols].isnull().sum(axis=1)
    case3_mask = null_counts >= 3
    if case3_mask.sum() > 0:
        for idx in df[case3_mask].index:
            dow = df.loc[idx, 'day_of_week']
            target_date = df.loc[idx, 'date']
            same_dow= df[(df['day_of_week'] == dow) & (df['date'] != target_date)].copy()
            same_dow['dist'] = (same_dow['date'] - target_date).abs()
            same_dow.sort_values('dist', inplace=True)
            before= same_dow[same_dow['date'] < target_date].head(3)
            after  = same_dow[same_dow['date'] > target_date].head(3)
            anchors = pd.concat([before, after])
            for col in metric_cols:
                if pd.isna(df.loc[idx, col]):
                    fill_val = anchors[col].mean()
                    if pd.notna(fill_val):
                        df.loc[idx, col] = fill_val
                        audit_imputed.append({
                            'col' : col,
                            'method': 'same-weekday avg (3+3)',
                            'count': 1,
                            'center': center_id
                        })
            df.loc[idx, 'imputed_flag'] = 1

    # Step 13: Calendar flags 
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['is_holiday'] = df['date'].dt.date.apply(lambda d: d in us_holidays).astype(int)

    #  Step 14: Merge staffing 
    staff_col = f'staffing_{center_id}'
    df = df.merge(staffing[['date', staff_col]], on='date', how='left')
    df.rename(columns={staff_col: 'daily_staffing'}, inplace=True)

    # Step 15: Drop raw date column 
    df.drop(columns=['date_raw'], inplace=True)

    # Step 16: Final null check
    remaining_nulls = df[metric_cols].isnull().sum()
    if remaining_nulls.sum() > 0:
        print(f"  [{center_id}] WARNING - nulls remaining:\n{remaining_nulls}")

    audit_log = {
        'dropped': pd.concat(audit_dropped) if audit_dropped else pd.DataFrame(),
        'imputed': pd.DataFrame(audit_imputed) if audit_imputed else pd.DataFrame()
    }

    return df, audit_log

print("Daily preprocessing function defined.")

Daily preprocessing function defined.


In [33]:
# Run Daily preprocessing for all centers and print results
daily_clean = {}
daily_audits = {}

for center in ['A', 'B', 'C', 'D']:
 
    print(f"Processing Daily - Center {center}")

    df_clean, audit = preprocess_daily(dfs[f'{center} - Daily'], center)
    daily_clean[center] = df_clean
    daily_audits[center] = audit

    print(f"  Shape after cleaning : {df_clean.shape}")
    print(f"  Date range : {df_clean['date'].min()} to {df_clean['date'].max()}")
    print(f"  Rows dropped: {len(audit['dropped'])}")
    print(f"  Rows imputed: {df_clean['imputed_flag'].sum()}")
    print(f"  Null check:\n{df_clean.isnull().sum()}")
    print(f"\n  Sample (5 rows):")
    print(df_clean.head())
    

Processing Daily — Center A
  Shape after cleaning : (712, 10)
  Date range : 2024-01-01 00:00:00 to 2025-12-31 00:00:00
  Rows dropped: 19
  Rows imputed: 22
  Null check:
date                0
call_volume         0
CCT                 0
service_level       0
abandon_rate        0
day_of_week         0
imputed_flag        0
is_weekend          0
is_holiday          0
daily_staffing    381
dtype: int64

  Sample (5 rows):
        date  call_volume     CCT  service_level  abandon_rate  day_of_week  \
0 2024-01-01       2147.0  302.45         0.9855        0.0037            0   
1 2024-01-02       7458.0  349.22         0.5213        0.1136            1   
2 2024-01-03       6882.0  331.07         0.7046        0.0432            2   
3 2024-01-04       6208.0  341.80         0.7200        0.0403            3   
4 2024-01-05       6190.0  334.56         0.8063        0.0291            4   

   imputed_flag  is_weekend  is_holiday  daily_staffing  
0             0           0           1  

In [34]:
# check staffing merge only for 2025 rows
daily_2025 = daily_clean['A'][daily_clean['A']['date'].dt.year == 2025]
print(f"2025 rows: {len(daily_2025)}")
print(f"2025 staffing nulls: {daily_2025['daily_staffing'].isnull().sum()}")
print(daily_2025.head())

2025 rows: 358
2025 staffing nulls: 27
          date  call_volume     CCT  service_level  abandon_rate  day_of_week  \
354 2025-01-01       2196.0  291.40         0.9504        0.0089            2   
355 2025-01-02       6711.0  307.38         0.9268        0.0101            3   
356 2025-01-03       6351.0  295.82         0.9476        0.0062            4   
357 2025-01-04       3910.0  291.55         0.9516        0.0126            5   
358 2025-01-05       1565.0  328.12         0.9061        0.0112            6   

     imputed_flag  is_weekend  is_holiday  daily_staffing  
354             0           0           1            47.0  
355             0           0           0            82.0  
356             0           0           0            92.0  
357             0           1           0            70.0  
358             0           1           0            40.0  


In [36]:
#Define the Interval preprocessing function
def preprocess_interval(df, center_id):
    df = df.copy()
    audit_dropped = []
    audit_imputed = []

    # Step 1: Rename columns 
    df.columns = ['month', 'day', 'interval', 'service_level',
                  'call_volume', 'abandoned_calls', 'abandon_rate', 'CCT']

    # --- Step 2: Build datetime from month + day + interval + year 2025 ---
    month_map = {
        'January':1,'February':2,'March':3,'April':4,
        'May':5,'June':6,'July':7,'August':8,
        'September':9,'October':10,'November':11,'December':12
    }
    df['month_num'] = df['month'].map(month_map)
    df['day']= pd.to_numeric(df['day'], errors='coerce')

    # interval is "HH:MM:SS" - split and take hour and minute only
    df['hour'] = df['interval'].astype(str).str.split(':').str[0]
    df['minute'] = df['interval'].astype(str).str.split(':').str[1]
    df['hour']= pd.to_numeric(df['hour'],   errors='coerce')
    df['minute'] = pd.to_numeric(df['minute'], errors='coerce')

    # build datetime
    def build_dt(row):
        try:
            return datetime(2025, int(row['month_num']), int(row['day']),
                            int(row['hour']), int(row['minute']))
        except:
            return pd.NaT

    df['datetime'] = df.apply(build_dt, axis=1)
    df['date']= pd.to_datetime(df['datetime'].dt.date)
    df['day_of_week'] = df['datetime'].dt.dayofweek

    # Step 3: Drop rows where datetime could not be built 
    bad_dt = df['datetime'].isna()
    if bad_dt.sum() > 0:
        audit_dropped.append(pd.DataFrame({
            'datetime': df.loc[bad_dt, 'interval'],
            'reason': 'unparseable datetime',
            'center': center_id,
            'sheet': 'interval'
        }))
        df = df[~bad_dt].copy()
        df.reset_index(drop=True, inplace=True)

    # slot_index AFTER bad rows dropped - no NaNs in hour/minute
    df['slot_index'] = df['hour'].astype(int) * 2 + df['minute'].astype(int) // 30

    # Step 4: Strip commas, cast numeric
    for col in ['call_volume', 'abandoned_calls']:
        df[col] = df[col].astype(str).str.replace(',', '', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['CCT'] = pd.to_numeric(df['CCT'], errors='coerce')

    # Step 5: Service Level and Abandoned Rate already 0.0-1.0 -just cast 
    for col in ['service_level', 'abandon_rate']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Step 6: Sort by datetime 
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Step 7: Drop duplicate datetimes 
    dupes = df.duplicated(subset='datetime', keep='first')
    if dupes.sum() > 0:
        print(f"  [{center_id}] Dropping {dupes.sum()} duplicate datetime rows")
        df = df[~dupes].copy()
        df.reset_index(drop=True, inplace=True)

    metric_cols = ['call_volume', 'abandoned_calls', 'abandon_rate', 'service_level', 'CCT']

    # Step 8: Add imputed_flag 
    df['imputed_flag'] = 0

    # Step 9: Case 2 - all metrics null → fill from same slot 1-3 weeks prior 
    all_null = df[metric_cols].isnull().all(axis=1)
    for idx in df[all_null].index:
        filled = False
        for weeks_back in [1, 2, 3]:
            prior_dt  = df.loc[idx, 'datetime'] - timedelta(weeks=weeks_back)
            prior_row = df[df['datetime'] == prior_dt]
            if not prior_row.empty and not prior_row[metric_cols].isnull().all(axis=1).values[0]:
                for col in metric_cols:
                    df.loc[idx, col] = prior_row[col].values[0]
                df.loc[idx, 'imputed_flag'] = 1
                audit_imputed.append({
                    'datetime': df.loc[idx, 'datetime'],
                    'col'     : 'all metrics',
                    'method'  : f'same-slot {weeks_back}w prior',
                    'center'  : center_id
                })
                filled = True
                break
        if not filled:
            audit_dropped.append(pd.DataFrame({
                'datetime': [df.loc[idx, 'datetime']],
                'reason'  : 'all metrics null, no prior slot found',
                'center'  : center_id,
                'sheet'   : 'interval'
            }))
            df.loc[idx, 'imputed_flag'] = -1

    # drop rows that could not be filled
    df = df[df['imputed_flag'] != -1].copy()
    df.reset_index(drop=True, inplace=True)

    # Step 10: Case 3 - 1-4 metrics null → same slot 1-2 weeks prior
    null_counts= df[metric_cols].isnull().sum(axis=1)
    partial_null = (null_counts >= 1) & (null_counts < 5)
    for idx in df[partial_null].index:
        for col in metric_cols:
            if pd.isna(df.loc[idx, col]):
                for weeks_back in [1, 2]:
                    prior_dt  = df.loc[idx, 'datetime'] - timedelta(weeks=weeks_back)
                    prior_row = df[df['datetime'] == prior_dt]
                    if not prior_row.empty and pd.notna(prior_row[col].values[0]):
                        df.loc[idx, col] = prior_row[col].values[0]
                        df.loc[idx, 'imputed_flag'] = 1
                        break


    # Step 10b: Remaining nulls -fill with mean of same slot + same weekday 
    null_counts = df[metric_cols].isnull().sum(axis=1)
    still_null= null_counts >= 1
    if still_null.sum() > 0:
        for idx in df[still_null].index:
            slot = df.loc[idx, 'slot_index']
            dow  = df.loc[idx, 'day_of_week']
            # get all rows with same slot and same weekday, excluding current
            similar = df[
                (df['slot_index']  == slot) &
                (df['day_of_week'] == dow)  &
                (df.index!= idx)
            ]
            for col in metric_cols:
                if pd.isna(df.loc[idx, col]):
                    fill_val = similar[col].mean()
                    if pd.notna(fill_val):
                        df.loc[idx, col] = fill_val
                        df.loc[idx, 'imputed_flag'] = 1
   
    #Step 11: Calendar flags 
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['is_holiday'] = df['date'].dt.date.apply(lambda d: d in us_holidays).astype(int)

    # Step 12: Drop helper columns
    df.drop(columns=['month', 'day', 'interval', 'month_num',
                     'hour', 'minute'], inplace=True)

    # Step 13: Final null check 
    remaining_nulls = df[metric_cols].isnull().sum()
    if remaining_nulls.sum() > 0:
        print(f"  [{center_id}] WARNING  nulls remaining:\n{remaining_nulls}")

    audit_log = {
        'dropped': pd.concat(audit_dropped) if audit_dropped else pd.DataFrame(),
        'imputed': pd.DataFrame(audit_imputed) if audit_imputed else pd.DataFrame()
    }
    df['day_of_week'] = df['day_of_week'].astype(int) 

    return df, audit_log

print("Interval preprocessing function defined.")

Interval preprocessing function defined.


In [41]:
# Run Interval preprocessing for all centers and print results
interval_clean = {}
interval_audits = {}

for center in ['A', 'B', 'C', 'D']:
    print(f"Processing Interval - Center {center}")
    df_clean, audit = preprocess_interval(dfs[f'{center} - Interval'], center)
    interval_clean[center] = df_clean
    interval_audits[center] = audit

    print(f"  Shape after cleaning : {df_clean.shape}")
    print(f"  Date range : {df_clean['datetime'].min()} to {df_clean['datetime'].max()}")
    print(f"  Rows dropped: {len(audit['dropped'])}")
    print(f"  Rows imputed: {df_clean['imputed_flag'].sum()}")
    print(f"  Null check  :\n{df_clean.isnull().sum()}")
    print(f"\n  Sample (3 rows):")
    print(df_clean.head(3))

Processing Interval - Center A
  Shape after cleaning : (4076, 12)
  Date range : 2025-04-01 00:00:00 to 2025-06-30 23:30:00
  Rows dropped: 7
  Rows imputed: 195
  Null check  :
service_level      0
call_volume        0
abandoned_calls    0
abandon_rate       0
CCT                0
datetime           0
date               0
day_of_week        0
slot_index         0
imputed_flag       0
is_weekend         0
is_holiday         0
dtype: int64

  Sample (3 rows):
   service_level  call_volume  abandoned_calls  abandon_rate     CCT  \
0            1.0          5.0              0.0           0.0  137.60   
1            1.0          5.0              0.0           0.0  263.40   
2            1.0          4.0              0.0           0.0  333.25   

             datetime       date  day_of_week  slot_index  imputed_flag  \
0 2025-04-01 00:00:00 2025-04-01            1           0             0   
1 2025-04-01 00:30:00 2025-04-01            1           1             0   
2 2025-04-01 01:00:00 

/tmp/ipykernel_899/4004282979.py:161: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  'dropped': pd.concat(audit_dropped) if audit_dropped else pd.DataFrame(),


  Shape after cleaning : (4279, 12)
  Date range : 2025-04-01 00:00:00 to 2025-06-30 23:30:00
  Rows dropped: 14
  Rows imputed: 141
  Null check  :
service_level      0
call_volume        0
abandoned_calls    0
abandon_rate       0
CCT                0
datetime           0
date               0
day_of_week        0
slot_index         0
imputed_flag       0
is_weekend         0
is_holiday         0
dtype: int64

  Sample (3 rows):
   service_level  call_volume  abandoned_calls  abandon_rate     CCT  \
0         1.0000         22.0              0.0           0.0  480.05   
1         0.9524         21.0              0.0           0.0  301.71   
2         1.0000          8.0              0.0           0.0  501.88   

             datetime       date  day_of_week  slot_index  imputed_flag  \
0 2025-04-01 00:00:00 2025-04-01            1           0             0   
1 2025-04-01 00:30:00 2025-04-01            1           1             0   
2 2025-04-01 01:00:00 2025-04-01            1       

In [38]:
# check how many rows had nulls in raw data before any processing
for center in ['A', 'B', 'C', 'D']:
    raw = dfs[f'{center} - Interval']
    null_rows = raw.isnull().any(axis=1).sum()
    print(f"Center {center} -raw rows with at least one null: {null_rows}")

Center A — raw rows with at least one null: 202
Center B — raw rows with at least one null: 155
Center C — raw rows with at least one null: 127
Center D — raw rows with at least one null: 359


In [39]:
metric_cols_raw = ['Service Level', 'Call Volume', 'Abandoned Calls', 'Abandoned Rate', 'CCT']

for center in ['A', 'B', 'C', 'D']:
    raw = dfs[f'{center} - Interval']
    null_rows = raw[metric_cols_raw].isnull().any(axis=1).sum()
    print(f"Center {center} -raw rows with at least one metric null: {null_rows}")

Center A — raw rows with at least one metric null: 202
Center B — raw rows with at least one metric null: 155
Center C — raw rows with at least one metric null: 127
Center D — raw rows with at least one metric null: 359


In [42]:
#Merge Daily into Interval and save all CSVs to S3
def save_to_s3(df, bucket, key):
    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue())
    print(f"  Saved: s3://{bucket}/{key}")

interval_merged = {}

for center in ['A', 'B', 'C', 'D']:
    print(f"Merging and saving - Center {center}")


    daily= daily_clean[center]
    intv = interval_clean[center]

    # cols to bring from daily into interval
    daily_cols = ['date', 'call_volume', 'CCT', 'service_level',
                  'abandon_rate', 'daily_staffing']
    daily_subset = daily[daily_cols].rename(columns={
        'call_volume' : 'daily_call_volume',
        'CCT': 'daily_CCT',
        'service_level': 'daily_service_level',
        'abandon_rate': 'daily_abandon_rate'
    })

    merged = intv.merge(daily_subset, on='date', how='left')
    interval_merged[center] = merged

    print(f"  Merged shape : {merged.shape}")
    print(f"  Columns      : {list(merged.columns)}")
    print(merged.head(3))

    # save to S3
    save_to_s3(daily[daily.columns],
               bucket, f'processed/center_{center}/daily_clean_{center}.csv')
    save_to_s3(merged,
               bucket, f'processed/center_{center}/interval_merged_{center}.csv')

    # save audit logs
    if not daily_audits[center]['dropped'].empty:
        save_to_s3(daily_audits[center]['dropped'],
                   bucket, f'processed/center_{center}/audit_dropped_daily_{center}.csv')
    if not interval_audits[center]['dropped'].empty:
        save_to_s3(interval_audits[center]['dropped'],
                   bucket, f'processed/center_{center}/audit_dropped_interval_{center}.csv')

print("\nAll centers preprocessed and saved to S3.")

Merging and saving - Center A
  Merged shape : (4076, 17)
  Columns      : ['service_level', 'call_volume', 'abandoned_calls', 'abandon_rate', 'CCT', 'datetime', 'date', 'day_of_week', 'slot_index', 'imputed_flag', 'is_weekend', 'is_holiday', 'daily_call_volume', 'daily_CCT', 'daily_service_level', 'daily_abandon_rate', 'daily_staffing']
   service_level  call_volume  abandoned_calls  abandon_rate     CCT  \
0            1.0          5.0              0.0           0.0  137.60   
1            1.0          5.0              0.0           0.0  263.40   
2            1.0          4.0              0.0           0.0  333.25   

             datetime       date  day_of_week  slot_index  imputed_flag  \
0 2025-04-01 00:00:00 2025-04-01            1           0             0   
1 2025-04-01 00:30:00 2025-04-01            1           1             0   
2 2025-04-01 01:00:00 2025-04-01            1           2             0   

   is_weekend  is_holiday  daily_call_volume  daily_CCT  daily_service